# LOAD LIBRARY

In [ ]:
# Library preprocessing text
import pandas as pd
from datetime import datetime
import os
import re

# Library bar chart dan wordcloud
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from wordcloud import WordCloud, STOPWORDS
from collections import Counter

# Library stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Library XLM-RoBERTa (analisis sentimen)
from transformers import pipeline

# PREPROCESSING

In [ ]:
project_name = ""
platform = ""  # 'x', 'ig', 'threads'

# ======================================================
# AUTO-RESOLVE PATHS DAN KOLOM
# ======================================================
PLATFORM_CONFIG = {
    'x': {'folder': 'x/tweets', 'suffix': 'tweets', 'id_col': 'tweet_id_str', 'label': 'Tweet'},
    'ig': {'folder': 'ig/posts', 'suffix': 'ig-posts', 'id_col': 'post_id_str', 'label': 'Post'},
    'threads': {'folder': 'threads/posts', 'suffix': 'threads-posts', 'id_col': 'post_id_str', 'label': 'Post'},
}

config = PLATFORM_CONFIG[platform]
file_name = f"{project_name}-{config['suffix']}.csv"
id_col = config['id_col']
data_label = config['label']

# SETUP FOLDER
base_folder = f"../../result/{project_name}/{config['folder']}"
output_folder = f"{base_folder}/preprocessing"
os.makedirs(output_folder, exist_ok=True)

data = pd.read_csv(f"{base_folder}/{file_name}")

# Konversi kolom created_at ke datetime aware dan ubah ke zona waktu Asia/Jakarta
data['created_at'] = pd.to_datetime(
    data['created_at'], format='%a %b %d %H:%M:%S %z %Y'
).dt.tz_convert('Asia/Jakarta').dt.strftime('%a %b %d %H:%M:%S %z %Y')

# Cek kolom tanggal & waktu ada. Jika belum, buat.
if 'tanggal' not in data.columns or 'waktu' not in data.columns:
    data['tanggal'] = pd.to_datetime(
        data['created_at'], format='%a %b %d %H:%M:%S %z %Y'
    ).dt.strftime('%Y-%m-%d')

    data['waktu'] = pd.to_datetime(
        data['created_at'], format='%a %b %d %H:%M:%S %z %Y'
    ).dt.strftime('%H:%M:%S')

# Dynamic column: pertahankan semua kolom asli, full_text di akhir
cols = [c for c in data.columns if c != 'full_text'] + ['full_text']
df = data[cols].copy()

df.drop_duplicates(subset=id_col, keep='last', inplace=True)
df = df[df['full_text'].notna() & (df['full_text'].str.strip() != "-") & (df['full_text'].str.strip() != "")]

df.info()
df.head(5)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======== FIX FORMAT TANGGAL ========
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")

# Ambil rentang otomatis
start_date = df["tanggal"].min().date()
end_date = df["tanggal"].max().date()
df_filtered = df[(df["tanggal"].dt.date >= start_date) & (df["tanggal"].dt.date <= end_date)]

# ======== Hitung jumlah data per bulan ========
df_filtered["bulan"] = df_filtered["tanggal"].dt.to_period("M").dt.to_timestamp()
monthly_counts = df_filtered.groupby("bulan").size().sort_index()

# ======== Cek jika data kosong ========
if monthly_counts.empty:
    print("Tidak ada data yang bisa ditampilkan dalam rentang waktu ini.")
else:
    # ======== Plot ========
    plt.figure(figsize=(12,6))
    plt.plot(monthly_counts.index, monthly_counts, color="#0D47A1", linewidth=2, marker="o")

    # ======== Format X-axis ========
    ax = plt.gca()
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%B"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=0, ha="center")

    # ======== Temukan puncak data ========
    max_value = monthly_counts.max()
    max_months = monthly_counts[monthly_counts == max_value].index

    if len(max_months) > 1:
        start_peak = max_months.min()
        end_peak = max_months.max()
        label_text = f"{max_value:,} {data_label}s ({start_peak.strftime('%B')} – {end_peak.strftime('%B %Y')})"
        peak_date = start_peak + (end_peak - start_peak) / 2
    else:
        peak_date = max_months[0]
        label_text = f"{max_value:,} {data_label}s ({peak_date.strftime('%B %Y')})"

    # ======== Atur batas atas Y agar label tidak keluar ========
    y_margin = max_value * 0.15
    plt.ylim(0, max_value + y_margin)

    # ======== Tambahkan label di dalam frame ========
    plt.text(
        peak_date, max_value - (y_margin * 0.3),
        label_text,
        ha="center", va="top",
        fontsize=10, color="white",
        bbox=dict(facecolor="#0D47A1", alpha=0.8, boxstyle="round,pad=0.3")
    )

    # ======== Styling ========
    plt.title(f"Trend {data_label} Monthly \"{project_name}\" Total: {int(len(df)):,} ({start_date.strftime('%B %Y')} – {end_date.strftime('%B %Y')})", fontsize=14, fontweight="bold")
    plt.xlabel("Months", fontsize=12)
    plt.ylabel(f"Number of {data_label}s", fontsize=12)
    plt.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()

    # ======== Save Figure ========
    plt.savefig(f"{output_folder}/{project_name}_trend_monthly.png", dpi=300, bbox_inches="tight")

    plt.show()

# ======== Simpan CSV Bulanan ========
monthly_df = monthly_counts.reset_index()
monthly_df.columns = ["bulan", "jumlah_data"]
monthly_df["bulan"] = monthly_df["bulan"].dt.strftime("%B %Y")
monthly_df.to_csv(f"{output_folder}/{project_name}_per_bulan.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ======== FIX FORMAT TANGGAL ========
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")

# Ambil rentang otomatis
start_date = df["tanggal"].min().date()
end_date = df["tanggal"].max().date()
df_filtered = df[(df["tanggal"].dt.date >= start_date) & (df["tanggal"].dt.date <= end_date)]

# Hitung jumlah data per tanggal
daily_counts = df_filtered.groupby("tanggal").size().sort_index()

# Hitung durasi total hari
total_days = (daily_counts.index.max() - daily_counts.index.min()).days

# Plot
plt.figure(figsize=(12,6))
plt.plot(daily_counts.index, daily_counts, color="#0D47A1", linewidth=2,)

# ======== Format X-axis adaptif ========
ax = plt.gca()

if total_days < 14:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
elif total_days < 31:
    week_labels = []
    current_month = None
    week_counter = 1
    last_label_date = None
    for date in daily_counts.index:
        if current_month != date.strftime("%b"):
            current_month = date.strftime("%b")
            week_counter = 1
        if last_label_date is None or (date - last_label_date).days >= 7:
            week_labels.append(f"{current_month} W{week_counter}")
            week_counter += 1
            last_label_date = date
        else:
            week_labels.append("")
    plt.xticks(daily_counts.index, week_labels, rotation=0, ha="center")
else:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%B"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())


# ======== Temukan puncak data ========
max_value = daily_counts.max()
max_dates = daily_counts[daily_counts == max_value].index

if len(max_dates) > 1:
    start_peak = max_dates.min()
    end_peak = max_dates.max()
    label_text = f"{max_value:,} {data_label}s ({start_peak.strftime('%d %B')} – {end_peak.strftime('%d %B')})"
    peak_date = start_peak + (end_peak - start_peak) / 2
else:
    peak_date = max_dates[0]
    label_text = f"{max_value:,} {data_label}s ({peak_date.strftime('%d %B %Y')})"

# ======== Atur batas atas Y agar label tidak keluar ========
y_margin = max_value * 0.15
plt.ylim(0, max_value + y_margin)

# ======== Tambahkan label di dalam frame ========
plt.text(
    peak_date, max_value - (y_margin * 0.3),
    label_text,
    ha="center", va="top",
    fontsize=10, color="white",
    bbox=dict(facecolor="#0D47A1", alpha=0.8, boxstyle="round,pad=0.3")
)

# ======== Styling ========
plt.title(f"Trend {data_label} Daily \"{project_name}\" Total: {int(len(df)):,} ({start_date.strftime('%d %B %Y')} – {end_date.strftime('%d %B %Y')})", fontsize=14, fontweight="bold")
plt.xlabel("Months", fontsize=12)
plt.ylabel(f"Number of {data_label}s", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()

# ======== Save Figure ========
plt.savefig(f"{output_folder}/{project_name}_trend_daily.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# CASE FOLDING
import unicodedata

def case_folding(tweet):
  if isinstance(tweet, str):
    tweet = unicodedata.normalize('NFKD', tweet)  # Unicode fancy → ASCII (𝗦𝗮𝘁𝗴𝗮𝘀 → Satgas)
    lowerCase = tweet.lower()
    return lowerCase

df['case_folding'] = df['full_text'].apply(case_folding)

df.head(5)

In [ ]:
# NORMALIZE
def normalize(text, kamus):
    import re

    if not isinstance(text, str):
        return "", [], [], []

    # Cleansing ringan
    text = re.sub(r'^rt[\s]+', ' ', text)
    text = re.sub(r'@[^\s]+', ' ', text)
    text = re.sub(r'#[^\s]+', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002702-\U000027B0"  # dingbats
        "\U000024C2-\U0001F251"  # enclosed chars
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    text = re.sub(r'&gt;', ' ', text)
    text = re.sub(r'&amp;|&', ' ', text)
    text = re.sub(r'amp[\s]+', ' ', text)
    text = re.sub(r'[^\w\s:%$.\-,!?/\"\\\']', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # Normalize slang words
    kata_baku = []
    kata_diganti = []
    kata_tidak_baku_hash = []
    sorted_patterns = sorted(kamus.keys(), key=len, reverse=True)

    for slang in sorted_patterns:
        pattern = re.compile(rf'\b{re.escape(slang)}\b')
        matches = pattern.findall(text)
        if matches:
            for match in matches:
                kata_diganti.append(match)
                kata_tidak_baku_hash.append(match)
            text = pattern.sub(kamus[slang], text)
            kata_baku.extend([kamus[slang]] * len(matches))

    return text, kata_baku, kata_diganti, kata_tidak_baku_hash


# Load kamus
kamus_kata = pd.read_csv("https://raw.githubusercontent.com/whdkelilauw/myfiles/main/kamus_normalize.csv")
kamus_kata['slang'] = kamus_kata['slang'].str.strip()
kamus_kata['formal'] = kamus_kata['formal'].str.strip()
kamus = dict(zip(kamus_kata['slang'], kamus_kata['formal']))

df['normalize'] = df['case_folding'].apply(lambda x: normalize(x, kamus)[0])

df.head(5)

In [ ]:
len(df)

In [ ]:
df_normalize = df.copy()

In [ ]:
df = df_normalize.copy()

In [ ]:
# === FILTER SPAM/IKLAN ===
SPAM_TERMS = [
    'gofood', 'go food', 'grabfood', 'grab food',
    'shopeefood', 'shopee food', 'sfood',
    'daget', 'voucher', 'promo', 'diskon'
]

SPAM_CONTACT = [
    'hub. kami', 'hub kami', 'hubungi kami'
]

def is_spam(text_normalize, text_casefold):
    if not isinstance(text_normalize, str) or not isinstance(text_casefold, str):
        return False
    for phrase in SPAM_CONTACT:
        if phrase in text_casefold:
            return True
    count = 0
    for term in SPAM_TERMS:
        if term in text_normalize:
            count += 1
    if re.search(r'08\d{8,12}', text_normalize):
        count += 1
    return count >= 3

before_spam = len(df)
spam_mask = df.apply(lambda r: is_spam(r['normalize'], r['case_folding']), axis=1)

df_spam = df[spam_mask][['full_text']].reset_index(drop=True)
df_spam.to_csv(f"{output_folder}/{project_name}_spam_removed.csv", index=False)
print(f"[OK] Spam tersimpan → {output_folder}/{project_name}_spam_removed.csv ({len(df_spam)} {data_label.lower()}s)")

df = df[~spam_mask].reset_index(drop=True)
print(f"Filter spam/iklan: {before_spam} → {len(df)} ({before_spam - len(df)} removed)")

# CLEANSING
def cleansing(tweet):
  if not isinstance(tweet, str):
      return ''
  tweet = tweet.encode('ascii', 'replace').decode('ascii')
  tweet = re.sub(r'\bb\d{1,3}\b|\d+', lambda m: m.group() if m.group()[0].isalpha() else '', tweet, flags=re.IGNORECASE)
  tweet = re.sub(r'^rt[\s]+', ' ', tweet)
  tweet = re.sub(r'@[^\s]+', ' ', tweet)
  tweet = re.sub(r'#[^\s]+', ' ', tweet)
  tweet = re.sub(r'http\S+|www\.\S+', '', tweet)
  emoji_pattern = re.compile(
      "["
      "\U0001F600-\U0001F64F"
      "\U0001F300-\U0001F5FF"
      "\U0001F680-\U0001F6FF"
      "\U0001F1E0-\U0001F1FF"
      "\U00002702-\U000027B0"
      "\U000024C2-\U0001F251"
      "]+", flags=re.UNICODE)
  tweet = emoji_pattern.sub(r'', tweet)
  tweet = re.sub(r'[^\w\s]', ' ', tweet)
  tweet = re.sub(r'&amp;|&', ' ', tweet)
  tweet = re.sub(r'amp[\s]+', ' ', tweet)
  tweet = re.sub(r'\s+', ' ', tweet).strip()
  return tweet

df['cleansing'] = df['normalize'].apply(cleansing)

df = df[df['cleansing'].notna() & (df['cleansing'].str.strip() != "-") & (df['cleansing'].str.strip() != "")]

# === FILTER KEYWORD CRAWLING ===
CRAWL_KEYWORDS = [
    "jampidsus", "febrie", 'febri', "adriansyah", "prabowo",
    "asabri", "krakatau steel", "don ritto", "don reton",
    "deponering", "pinangki", "kortastipidkor", "hotman", "paris",
    "boyamin", "maki", "nadiem", "makarim",
    "lhkpn", "mahfud", "panja"
]

before_kw = len(df)
keyword_mask = df['cleansing'].apply(
    lambda x: any(kw in x for kw in CRAWL_KEYWORDS)
)

df_no_keyword = df[~keyword_mask][['full_text']].reset_index(drop=True)
df_no_keyword.to_csv(f"{output_folder}/{project_name}_no_keyword_removed.csv", index=False)
print(f"[OK] No-keyword tersimpan → {output_folder}/{project_name}_no_keyword_removed.csv ({len(df_no_keyword)} {data_label.lower()}s)")

df = df[keyword_mask].reset_index(drop=True)
print(f"Filter keyword: {before_kw} → {len(df)} ({before_kw - len(df)} removed)")

df.head(5)

In [ ]:
# TOKENIZE
def tokenize(text):
  token = text.split()
  return token

df['tokenize'] = df['cleansing'].apply(tokenize)
df.head(5)

In [ ]:
# STOPWORD REMOVAL
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = stopwords.words('indonesian')

CUSTOM_STOPWORDS = [
    'a','b','c','d','e','f','g','h','i','j','k','l','m','n','o','p','q','r','s','t','u','v','w','x','y','z',
    'aww','awww','tu','tuh','uf','deh','dah','amp','je','rm','ufcc','ppv','of',
    'sih','eh','hm','hmm','hmmm','nya','kah','kan','kalo','kayak','doang','dll','tau',
    'ufufuf','lho','loh','lah','pon','je','pe','gs','ya','go','bro',
    'mah','lu','lo','loe','kau','ku','kuu','tau','ni','nih','gue','gua','gw',
    'kah','kahh','kak','kakk','kakak','kaka','kakkk','wkwk','wkwkwk','wkwkwkwk',
    'bang','abang','guy','guys','gaes','bawa','sok','pas','iya', 'wak'
]

stop_words.extend(CUSTOM_STOPWORDS)

sw = set(stop_words)

def stopword_removal(text):
  return [word for word in text if word not in sw]

df['stopword_removal'] = df['tokenize'].apply(stopword_removal)

df = df[df['stopword_removal'].notna() & (df['stopword_removal'].str.len() > 0)]
df.head(5)

In [ ]:
# STEMMING
factory = StemmerFactory()
stemmer = factory.create_stemmer()

custom_dict = {
  "kinerja": "kinerja",
  "pemerintah": "pemerintah",
  "pemerintahan": "pemerintah",
  "kejagung": "kejagung",
  "kejaksaan": "kejaksaan",
  "kuhap": "kuhap",
  'senilai': 'nilai',
  'dinilainya': 'nilai',
  'menilainya': 'nilai',
  'bernilai': 'nilai'
}

def stemming(text):
  if isinstance(text, list):
    stemmed_words = []
    for word in text:
      if word in custom_dict:
        stemmed_words.append(custom_dict[word])
      else:
        stemmed_words.append(stemmer.stem(word))
    return stemmed_words
  else:
    return ""

df['stemming'] = df['stopword_removal'].apply(stemming)
df.head(5)

In [ ]:
# SIMPAN HASIL
df.to_csv(f"{output_folder}/stemming_{file_name}", index=False, encoding="utf-8")

In [ ]:
# XLM-ROBERTA SENTIMENT ANALYSIS
xlm_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual"
xlm_pipe = pipeline('sentiment-analysis', model=xlm_name, tokenizer=xlm_name, truncation=True, max_length=512)

def predict_sentiment(texts, batch_size=32):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        results = xlm_pipe(batch)
        all_preds.extend([r['label'].lower() for r in results])
    return all_preds

# Strip leading ": " dari RT agar RT dan original dianggap sama saat dedup
if platform == 'x':
    df['_sentiment_key'] = df['normalize'].apply(lambda x: re.sub(r'^:\s*', '', x).strip() if isinstance(x, str) else x)
else:
    df['_sentiment_key'] = df['normalize']

# Unique text → predict → map balik
unique_texts = df['_sentiment_key'].drop_duplicates().reset_index(drop=True)
print(f"Total: {len(df)} {data_label.lower()}s → Unique (after dedup): {len(unique_texts)} texts")

unique_preds = predict_sentiment(unique_texts.tolist())
sentiment_map = dict(zip(unique_texts, unique_preds))
df['sentiment'] = df['_sentiment_key'].map(sentiment_map)

df.drop(columns=['_sentiment_key'], inplace=True)

print(f"[OK] Sentimen selesai")
print(df['sentiment'].value_counts().to_string())
df.head(5)

In [ ]:
# DISTRIBUSI SENTIMEN BARCHART
color_map = {
    "negative": "#F44336",
    "neutral": "#FFC107",
    "positive": "#4CAF50"
}

sentiment_counts = df["sentiment"].value_counts()

colors = [color_map[label] for label in sentiment_counts.index]

plt.figure(figsize=(7,4))
bars = sentiment_counts.plot(
    kind="bar",
    color=colors,
    alpha=0.8
)

for i, v in enumerate(sentiment_counts):
    pct = (v / len(df['sentiment'])) * 100
    plt.text(i, v, f"{v} | {pct:.2f}%", ha="center", va="bottom", fontsize=10)

plt.title(f"Distribusi Sentimen \"{project_name}\"", fontsize=14, fontweight="bold")
plt.xlabel("Kelas Sentimen", fontsize=12)
plt.ylabel(f"Jumlah {data_label}", fontsize=12)
plt.xticks(rotation=0)
plt.ylim(0, max(sentiment_counts) * 1.1)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.savefig(f"{output_folder}/{project_name}_barchart_sentiment.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# WORDCLOUD 
sw = set(STOPWORDS)
sw.update(CUSTOM_STOPWORDS)
sw.update([''])

SENTIMENT_COLORMAP = {
    "negative": "Reds",
    "positive": "Greens",
    "neutral":  "Greys"
}
DARK_BG = "#0f0f0f"
TITLE_COLOR = "white"

def generate_wordcloud_by_sentiment(df, sentiment_label, output_folder, project_name):
    subset = df[df['sentiment'] == sentiment_label]

    if subset.empty:
        print(f"[INFO] Tidak ada data untuk sentimen: {sentiment_label}")
        return

    tokens = subset['stemming'].explode().dropna()
    tokens = tokens[~tokens.isin(sw)]
    text = Counter(tokens)

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color=DARK_BG,
        contour_width=0,
        colormap=SENTIMENT_COLORMAP.get(sentiment_label, "gray"),
        max_words=800,
        random_state=42
    ).generate_from_frequencies(text)

    fig = plt.figure(figsize=(8,6), facecolor=DARK_BG)
    plt.imshow(wordcloud, interpolation='bilinear')

    plt.title(
        f"Wordcloud Sentimen {sentiment_label.capitalize()}",
        fontsize=14,
        fontweight="bold",
        color=TITLE_COLOR,
        pad=20
    )

    plt.axis('off')
    plt.tight_layout(rect=[0, 0, 1, 0.95])

    plt.savefig(
        f"{output_folder}/{project_name}_wordcloud_{sentiment_label}.png",
        dpi=300,
        bbox_inches="tight",
        facecolor=fig.get_facecolor()
    )
    plt.show()

for s in ['negative', 'neutral', 'positive']:
    generate_wordcloud_by_sentiment(
        df,
        sentiment_label=s,
        output_folder=output_folder,
        project_name=project_name
    )

In [ ]:
import pandas as pd
from collections import Counter
from wordcloud import STOPWORDS

# base + custom stopwords
sw = set(STOPWORDS)
sw.update(CUSTOM_STOPWORDS)
sw.update([''])

# explode list → filter stopwords
tokens = df['stemming'].explode().dropna()
tokens = tokens[~tokens.isin(sw)]

# hitung frekuensi
counter = Counter(tokens)
top_500 = counter.most_common(500)

# ubah urutan kolom → weight dulu, baru word
df_out = pd.DataFrame(
    [(freq, word) for word, freq in top_500],
    columns=['weight', 'word']
)

df_out.to_csv(f'{output_folder}/{project_name}_wordcloud_data.csv', index=False)

print(f"{project_name}_wordcloud_data.csv berhasil dibuat.")

In [ ]:
# SIMPAN HASIL
df.to_csv(f"{output_folder}/prep_{file_name}", index=False, encoding="utf-8")

# SPLIT DATA BY TANGGAL

### Cek Ideal Timeframe

In [ ]:
import os
import pandas as pd

# Gunakan project_name, config, file_name dari cell config di atas
prep_file = f"prep_{file_name}"

# Setup folder
base_split = f"{base_folder}/preprocessing"
os.makedirs(base_split, exist_ok=True)

# Load data
data = pd.read_csv(
    f"{base_split}/{prep_file}",
    parse_dates=['tanggal']
)

# Pastikan kolom tanggal sudah datetime
data['tanggal'] = pd.to_datetime(data['tanggal'])

# hitung jumlah data per hari
daily = data.groupby('tanggal').size().reset_index(name='count')
daily = daily.sort_values('tanggal').reset_index(drop=True)

# cari peak
peak_row = daily.loc[daily['count'].idxmax()]
peak_date = peak_row['tanggal']
peak_idx = daily.index[daily['tanggal'] == peak_date][0]

# tentukan window (jumlah hari sebelum dan sesudah peak_date)
window = 7

# === CARI RISE STARTS ===
rise_date = None
start_idx = max(0, peak_idx - window)

for i in range(start_idx, peak_idx - 2):
    if (daily.loc[i+1, 'count'] > daily.loc[i, 'count'] and
        daily.loc[i+2, 'count'] > daily.loc[i+1, 'count']):
        rise_date = daily.loc[i, 'tanggal']
        break

# === CARI FALL STARTS === 
after = daily[daily['tanggal'] > peak_date].copy()
after = after.reset_index(drop=True)

fall_date = None
turning_date = None

for i in range(1, len(after)):
    prev = after.loc[i-1, 'count']
    curr = after.loc[i, 'count']

    if fall_date is None and curr < prev:
        fall_date = after.loc[i, 'tanggal']

    if fall_date is not None and curr > prev:
        turning_date = after.loc[i, 'tanggal']
        break


since = (rise_date if rise_date is not None else peak_date).date()
until = (turning_date if turning_date is not None else peak_date).date()
print("Peak date:", peak_date.date())
print("Rise starts:", since)
print("Fall starts:", fall_date.date() if fall_date else None)
print("Fall ends (turning):", until)
print("=" * 60)
print(f"IDEAL TIMEFRAME = | since: {since} | until: {until} |")
print("=" * 60)

### Split Data

In [ ]:
# Input tanggal
since = "2026-01-18"
until = "2026-01-24"

# Filter data berdasarkan rentang tanggal
mask = (data['tanggal'] >= since) & (data['tanggal'] <= until)
filtered = data.loc[mask]

# Save ke CSV
output_path = f"{base_split}/{project_name}_{since}_{until}_{config['suffix']}.csv"
filtered.to_csv(output_path, index=False)

print("Saved:", output_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = pd.read_csv(output_path)

# ======== FIX FORMAT TANGGAL ========
df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")

# Ambil rentang otomatis
start_date = df["tanggal"].min().date()
end_date = df["tanggal"].max().date()
df_filtered = df[(df["tanggal"].dt.date >= start_date) & (df["tanggal"].dt.date <= end_date)]

# Hitung jumlah data per tanggal
daily_counts = df_filtered.groupby("tanggal").size().sort_index()

# Hitung durasi total hari
total_days = (daily_counts.index.max() - daily_counts.index.min()).days

# Plot
plt.figure(figsize=(12,6))
plt.plot(daily_counts.index, daily_counts, color="#0D47A1", linewidth=2)

# ======== Format X-axis adaptif ========
ax = plt.gca()

if total_days < 14:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
elif total_days < 31:
    week_labels = []
    current_month = None
    week_counter = 1
    last_label_date = None
    for date in daily_counts.index:
        if current_month != date.strftime("%b"):
            current_month = date.strftime("%b")
            week_counter = 1
        if last_label_date is None or (date - last_label_date).days >= 7:
            week_labels.append(f"{current_month} W{week_counter}")
            week_counter += 1
            last_label_date = date
        else:
            week_labels.append("")
    plt.xticks(daily_counts.index, week_labels, rotation=45, ha="center")
else:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%B"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())

# ======== Temukan puncak data ========
max_value = daily_counts.max()
max_dates = daily_counts[daily_counts == max_value].index

if len(max_dates) > 1:
    start_peak = max_dates.min()
    end_peak = max_dates.max()
    label_text = f"{max_value:,} {data_label}s ({start_peak.strftime('%d %B')} – {end_peak.strftime('%d %B')})"
    peak_date = start_peak + (end_peak - start_peak) / 2
else:
    peak_date = max_dates[0]
    label_text = f"{max_value:,} {data_label}s ({peak_date.strftime('%d %B %Y')})"

# ======== Atur batas atas Y agar label tidak keluar ========
y_margin = max_value * 0.15
plt.ylim(0, max_value + y_margin)

# ======== Tambahkan label di dalam frame ========
plt.text(
    peak_date, max_value - (y_margin * 0.3),
    label_text,
    ha="center", va="top",
    fontsize=10, color="white",
    bbox=dict(facecolor="#0D47A1", alpha=0.8, boxstyle="round,pad=0.3")
)

# ======== Styling ========
plt.title(f"Trend {data_label} ({start_date.strftime('%d %B')} – {end_date.strftime('%d %B %Y')})", fontsize=14, fontweight="bold")
plt.xlabel("Dates", fontsize=12)
plt.ylabel(f"Number of {data_label}s", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()

# ======== Save Figure ========
plt.savefig(f"{base_split}/trend_{since}_{until}.png", dpi=300, bbox_inches="tight")

plt.show()